In [0]:
%pip install databricks-sql-connector databricks-openai
dbutils.library.restartPython()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 113.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 119.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 59.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 57.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 108.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 111.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 606.1/606.1 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 76.5 MB/s eta 0:00:00
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.12.2
    Not uninstalling typing-extensions at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-ca79d3d6-f0fa-4058-80ae-8a16b86db5a9
    Can't uninstall 'typing_extensions'. No files were found to uninstall.
  Attem

In [0]:
# Imports
from databricks import sql
from databricks_openai import DatabricksOpenAI
import mlflow
import json
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

# Create OpenAI client for Databricks
client = DatabricksOpenAI()

In [0]:
# Conexión al SQL Warehouse
from databricks.sdk.core import Config

config = Config()
auth_dict = config.authenticate()
access_token = auth_dict['Authorization'].replace('Bearer ', '')

connection = sql.connect(
    server_hostname=config.host.replace('https://', ''),
    http_path="/sql/1.0/warehouses/45c1b4212424c665",
    access_token=access_token
)

In [0]:
def run_sql(query):
    try:
        with connection.cursor() as c:
            c.execute(query)
            result = c.fetchall()
        return result
    except Exception as e:
        return f"Error ejecutando SQL: {e}"

def get_available_tables():
    """Dynamically fetch all tables and their schemas from workspace.default"""
    try:
        query = """
        SELECT 
            table_name,
            column_name,
            data_type
        FROM workspace.information_schema.columns
        WHERE table_schema = 'default'
        ORDER BY table_name, ordinal_position
        """
        with connection.cursor() as c:
            c.execute(query)
            columns_info = c.fetchall()
        
        # Build schema information
        tables = {}
        for table_name, column_name, data_type in columns_info:
            if table_name not in tables:
                tables[table_name] = []
            tables[table_name].append(f"       - {column_name} ({data_type})")
        
        # Format as string
        schema_text = "Tablas disponibles en workspace.default:\n\n"
        for i, (table_name, columns) in enumerate(tables.items(), 1):
            schema_text += f"    {i}. workspace.default.{table_name}\n"
            schema_text += "\n".join(columns) + "\n\n"
        
        return schema_text
    except Exception as e:
        return f"Error obteniendo esquema: {e}"

In [0]:
def generate_sql(question):
    # Dynamically fetch available tables and their schemas
    schema_info = get_available_tables()
    
    prompt = f"""
    Eres un experto en SQL para Databricks.
    Convierte esta pregunta en una consulta SQL válida usando SOLAMENTE las tablas disponibles a continuación.
    
    {schema_info}
    
    IMPORTANTE:
    - USA SOLO las tablas y columnas listadas arriba
    - NO inventes nombres de tablas o columnas
    - Devuelve SOLO el SQL, sin explicaciones ni markdown
    
    Pregunta: {question}
    """

    response = client.chat.completions.create(
        model="databricks-meta-llama-3-3-70b-instruct",
        messages=[{"role": "user", "content": prompt}]
    )

    sql_query = response.choices[0].message.content
    
    # Remove markdown code block formatting if present
    sql_query = sql_query.strip()
    if sql_query.startswith('```'):
        # Remove opening ```sql or ```
        sql_query = sql_query.split('\n', 1)[1] if '\n' in sql_query else sql_query[3:]
    if sql_query.endswith('```'):
        # Remove closing ```
        sql_query = sql_query.rsplit('```', 1)[0]
    
    return sql_query.strip()

In [0]:
def generate_answer(question, sql_result):
    prompt = f"""
    Responde en lenguaje natural a esta pregunta usando los siguientes resultados SQL:

    Pregunta: {question}
    Resultados: {sql_result}

    Da una respuesta clara y concisa.
    """

    response = client.chat.completions.create(
        model="databricks-meta-llama-3-3-70b-instruct",
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content

In [0]:
# Widgets del chat
input_box = widgets.Text(
    value='',
    placeholder='Escribe tu pregunta...',
    description='Pregunta:',
    layout=widgets.Layout(width='80%')
)

button = widgets.Button(
    description='Enviar',
    button_style='primary'
)

output_area = widgets.Output()

def on_button_click(b):
    with output_area:
        clear_output()
        question = input_box.value
        
        display(Markdown(f"### ❓ Pregunta\n{question}"))
        
        # 1. Generar SQL
        sql_query = generate_sql(question)
        display(Markdown(f"**SQL generado:**\n```sql\n{sql_query}\n```"))
        
        # 2. Ejecutar SQL
        result = run_sql(sql_query)
        display(Markdown(f"**Resultado SQL:**\n```\n{result}\n```"))
        
        # 3. Respuesta final
        answer = generate_answer(question, result)
        display(Markdown(f"### ✅ Respuesta\n{answer}"))

button.on_click(on_button_click)

display(input_box, button, output_area)


Text(value='', description='Pregunta:', layout=Layout(width='80%'), placeholder='Escribe tu pregunta...')

Button(button_style='primary', description='Enviar', style=ButtonStyle())

Output()